**1. load and first look**

load and inspect

In [1]:
import numpy as np
import pandas as pd


In [8]:
df=pd.read_csv(r"D:\numpy and pandas project\Netflix movies and tv shows project\netflix_titles.csv")
df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


 Count missing values

In [ ]:
missing=df.isnull().sum()
missing

show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

In [16]:
missing_pct=(df.isnull().sum()/len(df)*100).round(2)
missing_pct

show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

count movies vs tv shows


In [47]:
counts=df['type'].value_counts()
pct=(df['type'].value_counts()/len(df)*100).round(2)
print('counts: \n',counts,'\npct: \n',pct)


counts: 
 type
Movie      6131
TV Show    2676
Name: count, dtype: int64 
pct: 
 type
Movie      69.62
TV Show    30.38
Name: count, dtype: float64


**2. data cleaning**

fill the missing director and cast

In [101]:
df['director']=df['director'].fillna('Unknown')
df['cast']=df['cast'].fillna('Unknown')
df['country']=df['country'].fillna('Unknown')
df['rating']=df['rating'].fillna('Unknown')
df.isnull().sum()

show_id             0
type                0
title               0
director            0
cast                0
country             0
date_added          0
release_year        0
rating              0
duration            3
listed_in           0
description         0
year_added          0
mounth_added        0
duration_int        3
duration_unit       3
duration_norm    7967
dtype: int64

convert date_added to datetime

In [103]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())

df['year_added']=df['date_added'].dt.year
df['month_added']=df['date_added'].dt.month

AttributeError: Can only use .str accessor with string values!

clean the duration column

In [64]:
df['duration_int']=df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit']=df['duration'].str.extract(r'[0-9]+(.*)')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   show_id        8807 non-null   object        
 1   type           8807 non-null   object        
 2   title          8807 non-null   object        
 3   director       8807 non-null   object        
 4   cast           8807 non-null   object        
 5   country        7976 non-null   object        
 6   date_added     8797 non-null   datetime64[ns]
 7   release_year   8807 non-null   int64         
 8   rating         8803 non-null   object        
 9   duration       8804 non-null   object        
 10  listed_in      8807 non-null   object        
 11  description    8807 non-null   object        
 12  year_added     8797 non-null   float64       
 13  mounth_added   8797 non-null   float64       
 14  duration_int   8804 non-null   float64       
 15  duration_unit  8804 n

drop rows with missing date_added

In [69]:
rows_before=len(df)
df.dropna(subset=['date_added'],inplace=True)
print('Removed:',rows_before-len(df))


Removed: 0


**3. string operations**

split genres and find top 10

In [104]:
genres=df['listed_in'].str.split(', ').explode()
top10=genres.value_counts().head(10)
print(top10)

listed_in
International Movies      2543
Dramas                    2317
Comedies                  1580
International TV Shows    1127
Action & Adventure         817
Documentaries              794
Independent Movies         745
TV Dramas                  662
Romantic Movies            588
Thrillers                  549
Name: count, dtype: int64


find the top 10 countary by content

In [76]:
rows_before=len(df)
print('rows_before',rows_before)
countries=df['country'].str.split(',').explode()
countries_filter=df.dropna(subset=['country'],inplace=True)
rows_after=len(df)
print('rows_after',rows_after)
c_top10=countries.value_counts().head(10)
print("c_top10:",c_top10)

rows_before 8797
rows_after 7967
c_top10: country
United States     3205
India             1008
United Kingdom     627
 United States     479
Canada             271
Japan              258
France             212
South Korea        211
Spain              181
 France            181
Name: count, dtype: int64


In [82]:
month_counts=df['mounth_added'].value_counts().sort_index()
month_names={1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
month_counts.index=month_counts.index.map(month_names)
print(month_counts)

mounth_added
Jan    701
Feb    541
Mar    686
Apr    694
May    545
Jun    633
Jul    708
Aug    667
Sep    661
Oct    701
Nov    671
Dec    759
Name: count, dtype: int64


**4. Analysis with groupby**

content added per year

In [92]:
df.groupby('year_added')['title'].count().sort_values(ascending=False)


year_added
2019.0    1858
2020.0    1771
2018.0    1530
2021.0    1140
2017.0    1123
2016.0     410
2015.0      79
2014.0      24
2011.0      13
2013.0      11
2012.0       3
2008.0       2
2009.0       2
2010.0       1
Name: title, dtype: int64

Average movie duration using NumPY

In [95]:
movies=df[df['type']=='Movie']['duration_int'].dropna().values
mean_dur=np.mean(movies).round(2)
median_dur=np.median(movies)
std_dur=np.std(movies).round(2)
print(f"Mean: {mean_dur} min, Median:{median_dur} min,std:{std_dur}")

Mean: 100.52 min, Median:99.0 min,std:27.12


Rating distribution by type

In [96]:
df.groupby(['type','rating'])['title'].count().unstack(fill_value=0)

rating,66 min,74 min,84 min,G,NC-17,NR,PG,PG-13,R,TV-14,TV-G,TV-MA,TV-PG,TV-Y,TV-Y7,TV-Y7-FV,UR
type,,,,,,,,,,,,,,,,,
Movie,1,1,1,41,3,75,281,482,787,1307,109,1924,504,84,83,4,3
TV Show,0,0,0,0,0,4,0,0,1,620,81,1005,267,143,152,1,0


**NumPY deep dive+pipeline**

Normalize movie duration

In [99]:
def minmax(arr):
    return (arr-np.min(arr))/(np.max(arr)-np.min(arr))
movie_mask=df['type']=='Movie'
df.loc[movie_mask,'duration_norm']=minmax(
    df.loc[movie_mask,'duration_int'].values
)

Finnd most productive directors using NumPY

In [105]:
dir_counts=df[df['director']!='Unknown']['director'].value_counts()
arr=dir_counts.values
print('Mean titles per director:',np.mean(arr).round(2))
print('Max title by one director:',np.max(arr))
print('Most productive:',dir_counts.index[0])

Mean titles per director: 1.34
Max title by one director: 18
Most productive: Raúl Campos, Jan Suter


Final clean_netflix() pipeline function

In [1]:
def clean_netflix(df):
    df = df.copy()
    df['director'] = df['director'].fillna('Unknown')
    df['cast'] = df['cast'].fillna('Unknown')
    df['country'] = df['country'].fillna('Unknown')
    df['rating'] = df['rating'].fillna('Unknown')
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
    df['year_added'] = df['date_added'].dt.year
    df['month_added'] = df['date_added'].dt.month
    df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)
    df['duration_unit'] = df['duration'].str.extract(r'[0-9]+(.*)')
    rows_before = len(df)
    df.dropna(subset=['date_added'], inplace=True)
    print("Removed:", rows_before - len(df))
    return df

clean_df = clean_netflix(pd.read_csv(r"D:\numpy and pandas project\Netflix movies and tv shows project\netflix_titles.csv"))
print(clean_df.shape)
print(clean_df.dtypes)

NameError: name 'pd' is not defined